# ASA-Transformer vs Dense Transformer — Block-ASA GPU Benchmark (Kaggle T4 x2)

**Adaptive Selective Attention (Block-ASA)** — Buenahora Ormaza (2026)  
Benchmark escalado: **1024 tokens**, **~500K params**, GPU T4 x2.  

### Optimizaciones Block-ASA activas
| # | Optimización | Impacto |
|---|---|---|
| 1 | Pass-1 a nivel de bloques ($C=16$) | Reduce longitud de routing $N \rightarrow N/C$ (1024 $\rightarrow$ 64 bloques) |
| 2 | Almacenamiento de índices de bloque | Reduce tensores de índices por $16\times$ y evita copia de clave por token |
| 3 | Pass-2 via Block-Sparse SDPA | Ejecuta la atención sobre bloques contiguos a velocidad máxima de Tensor Cores |
| 4 | Pass-1 sin gradiente (`no_grad`) | Libera la matriz de routing $O(N_{blocks}^2)$ inmediatamente |
| 5 | Padding automático a múltiplo de $C$ | Garantiza que secuencias de cualquier longitud ejecuten por la ruta rápida de bloques |

In [ ]:
# ── Cell 1: Environment ──────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pymbbo'])

import math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, List

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  VRAM={p.total_memory/1e9:.1f} GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active device: {DEVICE}')

# SDPA / FlashAttention-2 check
if torch.cuda.is_available() and hasattr(F, 'scaled_dot_product_attention'):
    _q = torch.randn(2, 4, 64, 32, device='cuda')
    _ = F.scaled_dot_product_attention(_q, _q, _q, is_causal=True)
    del _q; torch.cuda.empty_cache()
    print('[OK] FlashAttention-2 (SDPA) active')

In [ ]:
# ── Cell 2: Token-level Gather Helpers ───────────────────────────────
def _gather_kv_token(k: torch.Tensor, v: torch.Tensor, sel: torch.Tensor):
    B, H, N_total, D = k.shape
    N, A = sel.shape[1], sel.shape[2]
    k_2d = k.reshape(B * H, N_total, D)
    v_2d = v.reshape(B * H, N_total, D)
    sel_bh = sel.unsqueeze(1).expand(B, H, N, A).reshape(B * H, N * A)
    bh_idx = torch.arange(B * H, device=k.device).unsqueeze(1)
    k_gath = k_2d[bh_idx, sel_bh].reshape(B, H, N, A, D)
    v_gath = v_2d[bh_idx, sel_bh].reshape(B, H, N, A, D)
    return k_gath, v_gath

def _pass2_token_attention(q, k_gath, v_gath, scale):
    B, H, N, A, D = k_gath.shape
    if hasattr(F, "scaled_dot_product_attention") and q.is_cuda:
        q_s = q.reshape(B * H * N, 1, D)
        k_s = k_gath.reshape(B * H * N, A, D)
        v_s = v_gath.reshape(B * H * N, A, D)
        return F.scaled_dot_product_attention(q_s, k_s, v_s).reshape(B, H, N, D)
    scores = (q.unsqueeze(-2) * k_gath).sum(-1) * scale
    return (F.softmax(scores, dim=-1).unsqueeze(-1) * v_gath).sum(-2)

print('[OK] Token-level helpers defined.')

In [ ]:
# ── Cell 3: Core ASA Attention Module (Block + Token) ────────────────
class AdaptiveSelectiveAttention(nn.Module):
    """
    Adaptive Selective Attention (ASA) — Buenahora Ormaza (2026).
    Includes Block-level ASA (block_size=16) and Token-level fallback.
    Pads sequence length to block boundary automatically.
    """
    def __init__(self, d_model: int, nhead: int, max_a: int = 64,
                 is_router: bool = True, block_size: int = 16,
                 dropout: float = 0.0, margin_delta: float = 0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.d_model      = d_model
        self.nhead        = nhead
        self.head_dim     = d_model // nhead
        self.max_a        = max_a
        self.is_router    = is_router
        self.block_size   = block_size
        self.scale        = 1.0 / math.sqrt(self.head_dim)
        self.margin_delta = margin_delta
        self.q_proj   = nn.Linear(d_model, d_model)
        self.k_proj   = nn.Linear(d_model, d_model)
        self.v_proj   = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout  = nn.Dropout(dropout)

    def forward(self, x, selection_indices=None, max_a=None,
                return_aux_loss=False, kv_cache=None):
        B, N_orig, _ = x.shape
        H, D         = self.nhead, self.head_dim
        C            = self.block_size

        if C > 1 and (N_orig % C != 0) and kv_cache is None:
            pad_len = (C - (N_orig % C)) % C
            x = F.pad(x, (0, 0, 0, pad_len))
        else:
            pad_len = 0

        B, N, _ = x.shape

        q = self.q_proj(x).view(B, N, H, D).transpose(1, 2)
        k = self.k_proj(x).view(B, N, H, D).transpose(1, 2)
        v = self.v_proj(x).view(B, N, H, D).transpose(1, 2)

        if kv_cache is not None:
            ck, cv = kv_cache
            k = torch.cat([ck, k], dim=2)
            v = torch.cat([cv, v], dim=2)
        new_kv  = (k, v)
        N_total = k.shape[2]
        budget  = max_a if max_a is not None else self.max_a

        # Full attention fallback
        if budget >= N_total:
            if hasattr(F, 'scaled_dot_product_attention') and not return_aux_loss and x.is_cuda:
                dp  = self.dropout.p if self.training else 0.0
                out = F.scaled_dot_product_attention(
                    q, k, v, is_causal=(N == N_total), dropout_p=dp)
            else:
                s = (q @ k.transpose(-2, -1)) * self.scale
                if N == N_total:
                    s = s + torch.triu(torch.full((N, N), float('-inf'), device=x.device), 1)[None, None]
                out = F.softmax(s, dim=-1) @ v
            out = self.out_proj(out.transpose(1, 2).contiguous().view(B, N, self.d_model))
            fi  = torch.arange(N_total, device=x.device).view(1, 1, -1).expand(B, N, -1)
            if pad_len > 0: out = out[:, :N_orig, :]
            return out, fi, None, new_kv

        aux_loss = None

        # ── BLOCK-ASA PATH (Fast GPU Path) ───────────────────────────
        if C > 1 and (N % C == 0) and (N_total % C == 0):
            N_blocks       = N // C
            N_total_blocks = N_total // C
            A_blocks       = max(1, budget // C)

            if self.is_router or selection_indices is None:
                with torch.no_grad():
                    q_blk_m = q.reshape(B, H, N_blocks, C, D).mean(dim=3)
                    k_blk_m = k.reshape(B, H, N_total_blocks, C, D).mean(dim=3)
                    raw_b   = (q_blk_m @ k_blk_m.transpose(-2, -1)) * self.scale
                    cm_b    = torch.triu(
                        torch.full((N_blocks, N_total_blocks), float('-inf'), device=x.device),
                        diagonal=N_total_blocks - N_blocks + 1)
                    imp_b   = F.softmax(raw_b + cm_b[None, None], dim=-1).mean(1)
                    imp_bm  = imp_b.masked_fill(cm_b[None] == float('-inf'), -1e9)
                    a_cap   = min(A_blocks, N_total_blocks)
                    _, top_b = torch.topk(imp_bm, k=a_cap, dim=-1, sorted=False)
                    self_b  = (torch.arange(N_total_blocks - N_blocks, N_total_blocks, device=x.device)
                                   .view(1, -1, 1).expand(B, N_blocks, 1))
                    selection_indices = torch.cat([top_b, self_b], dim=-1)

                if return_aux_loss:
                    q_blk_mg = q.reshape(B, H, N_blocks, C, D).mean(dim=3)
                    k_blk_mg = k.reshape(B, H, N_total_blocks, C, D).mean(dim=3)
                    cm_bg    = torch.triu(
                        torch.full((N_blocks, N_total_blocks), float('-inf'), device=x.device),
                        diagonal=N_total_blocks - N_blocks + 1)
                    imp_bg   = F.softmax((q_blk_mg @ k_blk_mg.transpose(-2,-1)) * self.scale + cm_bg[None,None], dim=-1).mean(1)
                    sel_sb   = torch.gather(imp_bg, -1, selection_indices)
                    min_selb = sel_sb.min(-1).values
                    validb   = cm_bg[None].expand(B, N_blocks, N_total_blocks) != float('-inf')
                    smaskb   = torch.zeros((B, N_blocks, N_total_blocks), dtype=torch.bool, device=x.device)
                    smaskb.scatter_(-1, selection_indices, True)
                    non_sb   = imp_bg.masked_fill(~(validb & ~smaskb), -1e9)
                    mnsb     = non_sb.max(-1).values
                    mnsb     = torch.where(mnsb == -1e9, torch.zeros_like(mnsb), mnsb)
                    aux_loss = F.relu(self.margin_delta - min_selb + mnsb).mean()

            num_sel_b = selection_indices.shape[-1]
            k_blk = k.reshape(B, H, N_total_blocks, C, D)
            v_blk = v.reshape(B, H, N_total_blocks, C, D)
            sel_bh = selection_indices.unsqueeze(1).expand(B, H, N_blocks, num_sel_b).reshape(B * H, N_blocks * num_sel_b)
            bh_idx = torch.arange(B * H, device=x.device).unsqueeze(1)

            k_2d = k_blk.reshape(B * H, N_total_blocks, C * D)
            v_2d = v_blk.reshape(B * H, N_total_blocks, C * D)
            k_gath_b = k_2d[bh_idx, sel_bh].reshape(B, H, N_blocks, num_sel_b, C, D)
            v_gath_b = v_2d[bh_idx, sel_bh].reshape(B, H, N_blocks, num_sel_b, C, D)

            k_ctx = k_gath_b.reshape(B * H * N_blocks, num_sel_b * C, D)
            v_ctx = v_gath_b.reshape(B * H * N_blocks, num_sel_b * C, D)
            q_ctx = q.reshape(B * H * N_blocks, C, D)

            if hasattr(F, 'scaled_dot_product_attention') and x.is_cuda:
                out_ctx = F.scaled_dot_product_attention(q_ctx, k_ctx, v_ctx)
            else:
                s_b = (q_ctx @ k_ctx.transpose(-2, -1)) * self.scale
                out_ctx = F.softmax(s_b, dim=-1) @ v_ctx

            out = out_ctx.reshape(B, H, N, D)
            out = self.out_proj(out.transpose(1, 2).contiguous().view(B, N, self.d_model))
            if pad_len > 0: out = out[:, :N_orig, :]
            return out, selection_indices, aux_loss, new_kv

        # ── TOKEN-ASA FALLBACK PATH ─────────────────────────────────
        if self.is_router or selection_indices is None:
            with torch.no_grad():
                raw = (q.detach() @ k.detach().transpose(-2, -1)) * self.scale
                cm  = torch.triu(torch.full((N, N_total), float('-inf'), device=x.device), 1)
                imp = F.softmax(raw + cm[None, None], dim=-1).mean(1)
                imp_m = imp.masked_fill(cm[None] == float('-inf'), -1e9)
                a_cap = min(budget, N_total)
                _, top_idx = torch.topk(imp_m, k=a_cap, dim=-1, sorted=False)
                self_idx  = (torch.arange(N_total - N, N_total, device=x.device)
                                 .view(1, -1, 1).expand(B, N, 1))
                selection_indices = torch.cat([top_idx, self_idx], dim=-1)

        k_gath, v_gath = _gather_kv_token(k, v, selection_indices)
        out = _pass2_token_attention(q, k_gath, v_gath, self.scale)
        out = self.out_proj(out.transpose(1, 2).contiguous().view(B, N, self.d_model))
        if pad_len > 0: out = out[:, :N_orig, :]
        return out, selection_indices, aux_loss, new_kv


class FFN(nn.Module):
    def __init__(self, d_model: int, ff_dim: Optional[int] = None, dropout: float = 0.0):
        super().__init__()
        ff_dim = ff_dim or d_model * 4
        self.net = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(ff_dim, d_model))
    def forward(self, x): return self.net(x)


class ASABlock(nn.Module):
    def __init__(self, d_model, nhead, max_a=64, is_router=True, block_size=16, dropout=0.0):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = AdaptiveSelectiveAttention(d_model, nhead, max_a, is_router, block_size, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = FFN(d_model, dropout=dropout)
    def forward(self, x, sel=None, max_a=None, ret_aux=False, kvc=None):
        a, sel, aux, kvc = self.attn(self.ln1(x), sel, max_a, ret_aux, kvc)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return x, sel, aux, kvc


class ASALayerGroup(nn.Module):
    def __init__(self, g, d_model, nhead, max_a=64, block_size=16, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            ASABlock(d_model, nhead, max_a, is_router=(i==0), block_size=block_size, dropout=dropout)
            for i in range(g)])
    def forward(self, x, max_a=None, ret_aux=False, g_kvc=None):
        shared_sel, aux_total, new_kvc = None, None, []
        for i, layer in enumerate(self.layers):
            x, sel, aux, kvc_new = layer(x, shared_sel, max_a, ret_aux, g_kvc[i] if g_kvc else None)
            if i == 0: shared_sel = sel
            if aux is not None: aux_total = aux if aux_total is None else aux_total + aux
            if kvc_new is not None: new_kvc.append(kvc_new)
        return x, aux_total, new_kvc


class ASATransformerGPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=6,
                 max_seq_len=1024, group_size=2, max_a=64, block_size=16, dropout=0.0):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        groups, rem  = [], num_layers
        while rem > 0:
            g = min(group_size, rem)
            groups.append(ASALayerGroup(g, d_model, nhead, max_a, block_size, dropout))
            rem -= g
        self.groups   = nn.ModuleList(groups)
        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size)

    def forward(self, x, max_a=None, return_aux_loss=False, kv_caches=None):
        B, N  = x.shape
        past  = kv_caches[0][0][0].shape[2] if (kv_caches and kv_caches[0]) else 0
        pos   = torch.arange(past, past + N, device=x.device).unsqueeze(0)
        h     = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        aux_t, new_kvc = None, []
        for i, grp in enumerate(self.groups):
            h, aux, gkvc = grp(h, max_a, return_aux_loss, kv_caches[i] if kv_caches else None)
            if aux is not None: aux_t = aux if aux_t is None else aux_t + aux
            if gkvc: new_kvc.append(gkvc)
        logits = self.lm_head(self.final_ln(h))
        return (logits, aux_t) if return_aux_loss else logits

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=60, max_a=None):
        self.eval(); dev = prompt.device; curr = prompt.clone()
        B, N = curr.shape
        h    = self.drop(self.tok_emb(curr) + self.pos_emb(torch.arange(N, device=dev).unsqueeze(0)))
        kvc  = []
        for grp in self.groups:
            h, _, gkvc = grp(h, max_a); kvc.append(gkvc)
        curr = torch.cat([curr, self.lm_head(self.final_ln(h))[:, -1, :].argmax(-1, keepdim=True)], 1)
        for _ in range(max_new_tokens - 1):
            if curr.size(1) >= self.max_seq_len: break
            pl  = curr.size(1) - 1
            h   = self.drop(self.tok_emb(curr[:,-1:]) + self.pos_emb(torch.tensor([[pl]], device=dev)))
            nkvc = []
            for i, grp in enumerate(self.groups):
                h, _, gkvc = grp(h, max_a, g_kvc=kvc[i]); nkvc.append(gkvc)
            kvc  = nkvc
            curr = torch.cat([curr, self.lm_head(self.final_ln(h))[:,-1,:].argmax(-1, keepdim=True)], 1)
        return curr


class DenseGPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=6,
                 max_seq_len=1024, dropout=0.0):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        enc = nn.TransformerEncoderLayer(d_model, nhead, d_model*4, dropout,
                                          batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers, enable_nested_tensor=False)
        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, N = x.shape
        h    = self.tok_emb(x) + self.pos_emb(torch.arange(N, device=x.device).unsqueeze(0))
        mask = nn.Transformer.generate_square_subsequent_mask(N, device=x.device)
        return self.lm_head(self.final_ln(self.transformer(h, mask=mask, is_causal=True)))

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=60):
        self.eval(); curr = prompt.clone()
        for _ in range(max_new_tokens):
            if curr.size(1) >= self.max_seq_len: break
            curr = torch.cat([curr, self.forward(curr)[:,-1,:].argmax(-1, keepdim=True)], 1)
        return curr


def count_params(m): return sum(p.numel() for p in m.parameters())
print('[OK] All model classes defined.')

In [ ]:
# ── Cell 4: Structured dependency dataset ────────────────────────────
VOCAB, SEQ_LEN = 16, 1024

def make_data(n=250, seq=SEQ_LEN):
    data  = np.random.choice([4,5,9,10,11], size=(n, seq + 1)).astype(np.int64)
    rules = [(1,2),(6,7),(3,8)]
    for i in range(n):
        tok, dep = rules[i % 3]
        data[i,:4]=tok; data[i,250:255]=dep; data[i,750:755]=dep
    return torch.from_numpy(data[:, :-1]), torch.from_numpy(data[:, 1:])

X_tr, Y_tr = make_data(200); X_te, Y_te = make_data(40)
print(f'Train: {X_tr.shape}   Test: {X_te.shape}   Vocab: {VOCAB}')

In [ ]:
# ── Cell 5: Build models ──────────────────────────────────────────────
D_MODEL, NHEAD, N_LAYERS, MAX_SEQ, GROUP = 128, 4, 4, SEQ_LEN, 2
BLOCK_SIZE = 16

dense  = DenseGPT(VOCAB, D_MODEL, NHEAD, N_LAYERS, MAX_SEQ)
asa32  = ASATransformerGPT(VOCAB, D_MODEL, NHEAD, N_LAYERS, MAX_SEQ, GROUP, max_a=32, block_size=BLOCK_SIZE)
asa128 = ASATransformerGPT(VOCAB, D_MODEL, NHEAD, N_LAYERS, MAX_SEQ, GROUP, max_a=128, block_size=BLOCK_SIZE)

print(f'Dense GPT                  : {count_params(dense):,} params')
print(f'Block-ASA GPT (max_a=32)   : {count_params(asa32):,} params')
print(f'Block-ASA GPT (max_a=128)  : {count_params(asa128):,} params')

In [ ]:
# ── Cell 6: Training + benchmark utilities ────────────────────────────
def _sync():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
def reset_vram():
    if DEVICE.type == 'cuda': torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def peak_vram_mb():
    if DEVICE.type != 'cuda': return 0.0
    _sync(); return torch.cuda.max_memory_allocated() / 1e6

def train_model(model, X, Y, *, epochs=3, bs=8, lr=3e-3, is_asa=False, max_a=None):
    model.train().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        perm = torch.randperm(len(X)); loss_s, nb = 0.0, 0
        t0 = time.perf_counter()
        for i in range(0, len(X), bs):
            bx = X[perm[i:i+bs]].to(DEVICE); by = Y[perm[i:i+bs]].to(DEVICE)
            opt.zero_grad()
            lg = model(bx, max_a=max_a) if is_asa else model(bx)
            loss = F.cross_entropy(lg.view(-1, VOCAB), by.reshape(-1))
            loss.backward(); opt.step()
            loss_s += loss.item(); nb += 1
        print(f'  Epoch {ep+1}/{epochs} [{time.perf_counter()-t0:.2f}s]  loss={loss_s/nb:.4f}')

def eval_ppl(model, X, Y, *, is_asa=False, max_a=None, bs=8):
    model.eval(); total, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(X), bs):
            lg = model(X[i:i+bs].to(DEVICE), max_a=max_a) if is_asa else model(X[i:i+bs].to(DEVICE))
            total += F.cross_entropy(lg.view(-1,VOCAB), Y[i:i+bs].to(DEVICE).reshape(-1)).item(); n += 1
    return math.exp(total / n)

def bench_gen(model, *, is_asa=False, max_a=None, gen_tok=60, runs=6):
    model.eval(); p = torch.tensor([[1,1,1,1]], dtype=torch.int64, device=DEVICE)
    with torch.no_grad():
        if is_asa: model.generate(p, max_new_tokens=5, max_a=max_a)
        else:      model.generate(p, max_new_tokens=5)
    _sync(); times = []
    for _ in range(runs):
        _sync(); t0 = time.perf_counter()
        with torch.no_grad():
            if is_asa: model.generate(p, max_new_tokens=gen_tok, max_a=max_a)
            else:      model.generate(p, max_new_tokens=gen_tok)
        _sync(); times.append(time.perf_counter() - t0)
    mu, std = float(np.mean(times)), float(np.std(times))
    return mu, std, gen_tok / mu

print('[OK] Utilities ready.')

In [ ]:
# ── Cell 7: Standard Dense GPT ───────────────────────────────────────
print('='*60,'\n  Standard Dense GPT\n','='*60)
reset_vram()
train_model(dense, X_tr, Y_tr, epochs=3, bs=8, is_asa=False)
vtr_d = peak_vram_mb(); print(f'  VRAM (train): {vtr_d:.1f} MB')
ppl_d = eval_ppl(dense, X_te, Y_te, is_asa=False)
reset_vram()
t_d, s_d, tps_d = bench_gen(dense, is_asa=False, gen_tok=60)
vgen_d = peak_vram_mb()
print(f'  Perplexity  : {ppl_d:.4f}\n  Gen time    : {t_d:.4f}s ±{s_d:.5f}\n  Tok/sec     : {tps_d:.2f}\n  VRAM (gen)  : {vgen_d:.1f} MB')
reset_vram(); torch.cuda.empty_cache()

In [ ]:
# ── Cell 8: Block-ASA GPT max_a=32 ───────────────────────────────────
print('='*60,'\n  Block-ASA GPT  max_a=32  (block_size=16)\n','='*60)
reset_vram()
train_model(asa32, X_tr, Y_tr, epochs=3, bs=8, is_asa=True, max_a=32)
vtr_32 = peak_vram_mb(); print(f'  VRAM (train): {vtr_32:.1f} MB')
ppl_32 = eval_ppl(asa32, X_te, Y_te, is_asa=True, max_a=32)
reset_vram()
t_32, s_32, tps_32 = bench_gen(asa32, is_asa=True, max_a=32, gen_tok=60)
vgen_32 = peak_vram_mb()
print(f'  Perplexity  : {ppl_32:.4f}\n  Gen time    : {t_32:.4f}s ±{s_32:.5f}\n  Tok/sec     : {tps_32:.2f}\n  VRAM (gen)  : {vgen_32:.1f} MB')
reset_vram(); torch.cuda.empty_cache()

In [ ]:
# ── Cell 9: Block-ASA GPT max_a=128 ──────────────────────────────────
print('='*60,'\n  Block-ASA GPT  max_a=128  (block_size=16)\n','='*60)
reset_vram()
train_model(asa128, X_tr, Y_tr, epochs=3, bs=8, is_asa=True, max_a=128)
vtr_128 = peak_vram_mb(); print(f'  VRAM (train): {vtr_128:.1f} MB')
ppl_128 = eval_ppl(asa128, X_te, Y_te, is_asa=True, max_a=128)
reset_vram()
t_128, s_128, tps_128 = bench_gen(asa128, is_asa=True, max_a=128, gen_tok=60)
vgen_128 = peak_vram_mb()
print(f'  Perplexity  : {ppl_128:.4f}\n  Gen time    : {t_128:.4f}s ±{s_128:.5f}\n  Tok/sec     : {tps_128:.2f}\n  VRAM (gen)  : {vgen_128:.1f} MB')
reset_vram(); torch.cuda.empty_cache()

In [ ]:
# ── Cell 10: Final comparative table ─────────────────────────────────
rows = [
    ('Standard Dense GPT',        count_params(dense),  ppl_d,   vtr_d,   t_d,   s_d,   tps_d,   vgen_d),
    ('Block-ASA GPT (max_a=32)',  count_params(asa32),   ppl_32,  vtr_32,  t_32,  s_32,  tps_32,  vgen_32),
    ('Block-ASA GPT (max_a=128)', count_params(asa128),  ppl_128, vtr_128, t_128, s_128, tps_128, vgen_128),
]
W = 124
print('='*W)
print('  BENCHMARK — 1024 TOKENS / ~500K PARAMS / GPU T4 x2  (Block-ASA block_size=16)')
print('='*W)
print(f"{'MODELO':<27} | {'PARAMS':>10} | {'PPL':>8} | {'VRAM-TR':>9} | {'GEN-T':>8} | {'STD':>7} | {'TOK/SEC':>9} | {'VRAM-GEN':>9}")
print('-'*W)
for name, p, ppl, vtr, t, std, tps, vgen in rows:
    print(f"{name:<27} | {p:>10,} | {ppl:>8.4f} | {vtr:>8.1f}M | {t:>8.4f}s | {std:>7.5f} | {tps:>9.2f} | {vgen:>8.1f}M")
print('='*W)

ref_tps, ref_vtr = rows[0][6], rows[0][3]
print('\nSpeedup tok/sec vs Dense GPT:')
for name, *_, tps, _ in rows: print(f'  {name:<27}: {tps/ref_tps:.3f}x')
print('\nVRAM-Training vs Dense GPT:')
for name, _, __, vtr, *_ in rows: print(f'  {name:<27}: {vtr:.1f} MB  ({(1-vtr/ref_vtr)*100:+.1f}%)')
print('\n[Done]')